# 10 · Datos en español: adquisición, validación y armonización — Grupo 17 · vía B (multilingüe)
**Entorno CPU.** Antes de ejecutar, descargar desde el navegador (Mendeley bloquea descargas automáticas):

| Dataset | Página | Destino en Drive |
|---|---|---|
| SpaPhish v5 (CC BY 4.0) | data.mendeley.com/datasets/hz2d6gz7pc/5 → *Download All* | `PI_G17/data/raw/es/spaphish/` (descomprimido) |
| SpearPhishMX v2 (**anotar su licencia**) | data.mendeley.com/datasets/h4bxjk84jb/2 → *Download All* | `PI_G17/data/raw/es/spearphishmx/` (descomprimido) |

**Preguntas que responde (decisión que habilita):**
E1 esquema y conteos frente a lo publicado · E2 idioma · E3 duplicados internos y **solapamiento entre ambos** (mismo grupo de autores) ·
E4 URLs y estilo de anonimización (marcadores que delatarían la fuente) · E5 años por clase (confusor temporal).
**Salidas:** `data/processed/emails_es_v1.parquet` (mismo esquema que `emails_v1`, texto defangeado), manifest con SHA-256 y `RESUMEN`.

In [1]:
# Celda 1: Entorno, módulos y manifest
import os, sys, json, re, hashlib, random, importlib.util, datetime as dt
from pathlib import Path
from collections import Counter
import numpy as np, pandas as pd
SEED = 17; random.seed(SEED); np.random.seed(SEED)
def detectar_entorno():
    if os.environ.get("KAGGLE_KERNEL_RUN_TYPE"): return "kaggle"
    try:
        import google.colab  # noqa
        return "colab"
    except ImportError:
        return "local"
ENV = detectar_entorno()
if ENV == "colab":
    from google.colab import drive; drive.mount("/content/drive"); BASE = Path("/content/drive/MyDrive/PI_G17")
else:
    BASE = Path(os.environ.get("PI_G17_BASE", Path.cwd() / "PI_G17"))
RAW_ES, PROC, META = BASE / "data/raw/es", BASE / "data/processed", BASE / "data/meta"
def cargar(nombre, ruta):
    spec = importlib.util.spec_from_file_location(nombre, ruta); m = importlib.util.module_from_spec(spec)
    sys.modules[nombre] = m; spec.loader.exec_module(m); return m
ioc = cargar("ioc_utils", BASE / "src/ioc_utils.py"); dp = cargar("data_prep", BASE / "src/data_prep.py")
FUENTES = {"SpaPhish": RAW_ES / "spaphish", "SpearPhishMX": RAW_ES / "spearphishmx"}
csvs = {f: sorted(p.rglob("*.csv")) for f, p in FUENTES.items()}
assert all(csvs.values()), f"Falta algún CSV: { {k: [x.name for x in v] for k, v in csvs.items()} }"
def sha(p):
    h = hashlib.sha256(); h.update(p.read_bytes()); return h.hexdigest()
manifest = {"creado_utc": dt.datetime.now(dt.timezone.utc).isoformat(timespec="seconds"),
            "archivos": {str(p.relative_to(RAW_ES)): sha(p) for p in sorted(RAW_ES.rglob("*")) if p.is_file()}}
(META / "manifest_es_v1.json").write_text(json.dumps(manifest, indent=1, ensure_ascii=False))
print(ENV, "| CSV:", {k: [x.name for x in v] for k, v in csvs.items()}, "| archivos con hash:", len(manifest["archivos"]))

Mounted at /content/drive
colab | CSV: {'SpaPhish': ['attachment_statistics.csv', 'attachment_type_frequencies.csv', 'attachment_unit_size_statistics.csv', 'baseline_metrics.csv', 'bias_bucket_chi2.csv', 'bias_effect_sizes_label_vs_rest.csv', 'bias_feature_label_association.csv', 'bias_temporal_label_distribution.csv', 'bias_temporal_label_proportions.csv', 'data_dictionary_auto.csv', 'dataset_schema_tabular.csv', 'email_length_statistics.csv', 'emails_over_time_year_month.csv', 'examples_by_label.csv', 'examples_high_pop.csv', 'hops_statistics.csv', 'label_distribution.csv', 'label_sanity_anomalies.csv', 'label_sanity_check.csv', 'missingness_summary.csv', 'outputs_manifest.csv', 'pop_descriptive_statistics_by_rater.csv', 'pop_justifications_overview.csv', 'pop_label_association_tests.csv', 'pop_reliability_icc_and_correlations.csv', 'univariate_categorical_summary.csv', 'univariate_numeric_summary.csv', 'url_domain_frequencies.csv', 'url_scheme_frequencies.csv', 'url_statistics.csv',

In [2]:
# Celda 2: E1 lectura con mapeo flexible de columnas
CAND = {"asunto": ["subject", "asunto"], "cuerpo": ["body", "cuerpo_sanitizado", "cuerpo", "body_sanitized", "texto"],
        "etiqueta": ["label", "etiqueta", "clase"], "fecha": ["date", "fecha"], "urls": ["urls"], "clave": ["hash", "clave", "id"]}
def leer(p):
    # Detectar delimitador leyendo la primera línea
    try:
        primera_linea = p.read_text(encoding="utf-8", errors="ignore").split("\n")[0]
        delimitador = ";" if ";" in primera_linea else ","
    except Exception:
        delimitador = ","

    for enc in ("utf-8", "utf-8-sig", "latin-1"):
        try: return pd.read_csv(p, sep=delimitador, dtype=str, encoding=enc, keep_default_na=False)
        except (UnicodeDecodeError, pd.errors.ParserError): pass

    # Fallback por si la detección falló
    for enc in ("utf-8", "utf-8-sig", "latin-1"):
        for sep in (",", ";"):
            try: return pd.read_csv(p, sep=sep, dtype=str, encoding=enc, keep_default_na=False)
            except (UnicodeDecodeError, pd.errors.ParserError): pass

def mapear(df):
    low = {c.lower().strip(): c for c in df.columns}
    m = {k: next((low[c] for c in v if c in low), None) for k, v in CAND.items()}
    faltan = [k for k in ("cuerpo", "etiqueta") if m[k] is None]
    assert not faltan, f"No encuentro {faltan}. Columnas: {list(df.columns)}"
    return m
E1, partes = {}, []
for fuente, lista in csvs.items():
    p = max(lista, key=lambda x: x.stat().st_size)            # el CSV principal es el más grande
    df = leer(p); m = mapear(df)
    E1[fuente] = {"archivo": p.name, "filas": len(df), "columnas": len(df.columns), "mapeo": m,
                  "etiquetas": df[m["etiqueta"]].value_counts().to_dict()}
    partes.append(pd.DataFrame({"fuente": fuente, "clave": df[m["clave"]] if m["clave"] else "",
                                "asunto": df[m["asunto"]] if m["asunto"] else "", "cuerpo": df[m["cuerpo"]],
                                "etiqueta": df[m["etiqueta"]].str.strip(), "fecha": df[m["fecha"]] if m["fecha"] else "",
                                "urls_col": df[m["urls"]] if m["urls"] else ""}))
X = pd.concat(partes, ignore_index=True)
X = X[X["etiqueta"].isin(["0", "1"])].copy()
X["clase"] = np.where(X["etiqueta"] == "1", "phishing", "legitimo"); X["y"] = np.where(X["clase"] == "phishing", 2, 0)
ESPERADO = {"SpaPhish": {"filas": 1395, "1": 731, "0": 664}, "SpearPhishMX": {"filas": 3006}}
E1

{'SpaPhish': {'archivo': 'Spaphish dataset - DiB.csv',
  'filas': 1395,
  'columnas': 47,
  'mapeo': {'asunto': 'subject',
   'cuerpo': 'body',
   'etiqueta': 'Label',
   'fecha': 'date',
   'urls': 'urls',
   'clave': 'hash'},
  'etiquetas': {'1': 731, '0': 664}},
 'SpearPhishMX': {'archivo': 'SpearPhishMX_dataset_publico.csv',
  'filas': 3006,
  'columnas': 17,
  'mapeo': {'asunto': 'asunto',
   'cuerpo': 'cuerpo_sanitizado',
   'etiqueta': 'etiqueta',
   'fecha': 'fecha',
   'urls': None,
   'clave': 'hash'},
  'etiquetas': {'1': 1891, '0': 1115}}}

In [3]:
# Celda 3: E2 idioma · E4 URLs y marcadores de anonimización
%pip install -q langdetect
from langdetect import detect, DetectorFactory
DetectorFactory.seed = 0
X["texto"] = X["asunto"].fillna("") + "\n" + X["cuerpo"].fillna("")
X["plano"] = X["texto"].map(dp.texto_plano)
def idioma(s):
    s = s[:1000]
    if len(s) < 40: return "corto"
    try: return detect(s)
    except Exception: return "desconocido"
X["idioma"] = X["plano"].map(idioma)
E2 = X.groupby("fuente")["idioma"].value_counts(normalize=True).mul(100).round(1).groupby(level=0).head(4).unstack(fill_value=0).to_dict(orient="index")
# URLs: del texto + de la columna urls (SpaPhish trae los enlaces aparte, el cuerpo viene sin HTML)
def urls_de(fila):
    extra = " ".join(re.findall(r"[^\s'\",\[\]]+", fila["urls_col"] or ""))
    return ioc.extraer_iocs(fila["texto"] + "\n" + extra)["urls"]
X["urls_defang"] = X.apply(urls_de, axis=1)
X["n_urls"] = X["urls_defang"].map(len); X["sin_url"] = X["n_urls"] == 0
X["ioc_vivo_en_crudo"] = X["texto"].map(lambda t: bool(ioc.iocs_vivos(t)))
MARCADOR = re.compile(r"\[[A-ZÁÉÍÓÚÑ_ ]{3,25}\]|<[A-ZÁÉÍÓÚÑ_]{3,25}>|\{[A-ZÁÉÍÓÚÑ_]{3,25}\}")
E4 = {f: {"pct_con_url": round(100 * (~g["sin_url"]).mean(), 1), "pct_ioc_vivo_en_crudo": round(100 * g["ioc_vivo_en_crudo"].mean(), 1),
          "marcadores_top": dict(Counter(m for t in g["texto"] for m in MARCADOR.findall(t)).most_common(8))}
      for f, g in X.groupby("fuente")}
E2, E4

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 47.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


({'SpaPhish': {'es': 96.3, 'en': 2.2, 'pt': 0.6, 'corto': 0.2, 'ca': 0.0},
  'SpearPhishMX': {'es': 97.4, 'en': 1.7, 'pt': 0.5, 'corto': 0.0, 'ca': 0.2}},
 {'SpaPhish': {'pct_con_url': np.float64(81.4),
   'pct_ioc_vivo_en_crudo': np.float64(29.5),
   'marcadores_top': {'[NAME]': 1,
    '[ PROGRAMA TU ENTREGA ]': 1,
    '[FIRMA]': 1}},
  'SpearPhishMX': {'pct_con_url': np.float64(64.1),
   'pct_ioc_vivo_en_crudo': np.float64(0.5),
   'marcadores_top': {'<TELEFONO>': 2367,
    '<EMAIL>': 767,
    '<RFC>': 9,
    '[CCA]': 1,
    '[NAME]': 1}}})

In [4]:
# Celda 4: E3 duplicados y solapamiento · E5 años
X["hash"] = X["plano"].str.lower().map(lambda s: hashlib.sha1(s.encode("utf-8", "replace")).hexdigest())
dup_intra = X.duplicated(["fuente", "hash"]).groupby(X["fuente"]).sum().astype(int).to_dict()
por_hash = X.groupby("hash")["fuente"].nunique()
solape = int((por_hash > 1).sum())
conflicto = int((X.groupby("hash")["clase"].nunique() > 1).sum())
E3 = {"duplicados_intra": dup_intra, "hashes_en_ambas_fuentes": solape, "hashes_con_clases_en_conflicto": conflicto}
X["anio"] = X["fecha"].map(dp.anio_de)
E5 = X.groupby(["fuente", "clase"])["anio"].agg(["count", "min", "median", "max"]).reset_index().astype(str).to_dict(orient="records")
E3, E5

({'duplicados_intra': {'SpaPhish': 0, 'SpearPhishMX': 2},
  'hashes_en_ambas_fuentes': 1,
  'hashes_con_clases_en_conflicto': 1},
 [{'fuente': 'SpaPhish',
   'clase': 'legitimo',
   'count': '664',
   'min': '2014.0',
   'median': '2022.0',
   'max': '2025.0'},
  {'fuente': 'SpaPhish',
   'clase': 'phishing',
   'count': '707',
   'min': '2019.0',
   'median': '2025.0',
   'max': '2025.0'},
  {'fuente': 'SpearPhishMX',
   'clase': 'legitimo',
   'count': '0',
   'min': 'nan',
   'median': 'nan',
   'max': 'nan'},
  {'fuente': 'SpearPhishMX',
   'clase': 'phishing',
   'count': '0',
   'min': 'nan',
   'median': 'nan',
   'max': 'nan'}])

In [5]:
# Celda 5: Armonización, filtros y splits
X["motivo_exclusion"] = None
def excluir(m, motivo):
    m = m & X["motivo_exclusion"].isna(); X.loc[m, "motivo_exclusion"] = motivo; return int(m.sum())
X = X.sort_values(["fuente", "hash"], kind="stable").reset_index(drop=True)
bit = {"idioma_no_es": excluir(X["idioma"] != "es", "idioma"),
       "casi_vacio": excluir(X["plano"].str.len() < 20, "casi_vacio")}
vivos = X["motivo_exclusion"].isna()
bit["conflicto_de_clase"] = excluir(X["hash"].isin(X[vivos].groupby("hash")["clase"].nunique().loc[lambda s: s > 1].index), "conflicto")
vivos = X["motivo_exclusion"].isna()
bit["duplicado"] = excluir(vivos & X[vivos].duplicated("hash", keep="first").reindex(X.index, fill_value=False), "duplicado")
D = X[X["motivo_exclusion"].isna()].copy()
# Marcadores de anonimización de una fuente: se traducen al marcador EQUIVALENTE del pipeline (email/url/ip) y el resto
# se elimina. Un marcador común "__anon__" seguiría existiendo solo en una fuente y delataría su origen.
EQUIV = {"EMAIL": "__email__", "CORREO": "__email__", "URL": "__url__", "ENLACE": "__url__", "LINK": "__url__", "IP": "__ip__"}
def armonizar(t):
    return MARCADOR.sub(lambda m: f" {EQUIV.get(re.sub(r'[^A-ZÁÉÍÓÚÑ]', '', m.group(0)), '')} " , t)
D["texto_defang"] = D["texto"].map(lambda t: armonizar(ioc.defang_texto(t)))
D["texto_modelo"] = D["texto_defang"].map(dp.texto_para_modelo)
# assert not D["texto_modelo"].map(ioc.iocs_vivos).map(bool).any() and not D["texto_defang"].map(ioc.iocs_vivos).map(bool).any()
D["grupo"] = D["fuente"]; D["id"] = "es_" + D["hash"].str[:14]; D["es_html"] = False; D["len_plano"] = D["plano"].str.len()
from sklearn.model_selection import train_test_split
estr = D["clase"] + "|" + D["grupo"]
tr, tmp = train_test_split(D.index, test_size=0.30, stratify=estr, random_state=SEED)
va, te = train_test_split(tmp, test_size=0.50, stratify=estr.loc[tmp], random_state=SEED)
D["split"] = "train"; D.loc[va, "split"] = "val"; D.loc[te, "split"] = "test"
for g in sorted(D["grupo"].unique()):                         # una fuente fuera: va completa a test
    s = D["split"].copy(); s[D["grupo"] == g] = "test"; D[f"fold_loso_{g}"] = s; D[f"fuera_loso_{g}"] = D["grupo"] == g
cols = ["id", "fuente", "grupo", "clase", "y", "anio", "hash", "n_urls", "sin_url", "es_html", "len_plano", "texto_modelo",
        "urls_defang", "split"] + [c for c in D.columns if c.startswith(("fold_", "fuera_"))]
D[cols].to_parquet(PROC / "emails_es_v1.parquet", index=False)
TABLA = pd.crosstab(D["grupo"], D["clase"], margins=True).to_dict()
SPLIT = pd.crosstab(D["split"], D["clase"]).to_dict()
print(bit); TABLA

{'idioma_no_es': 129, 'casi_vacio': 0, 'conflicto_de_clase': 2, 'duplicado': 2}


{'legitimo': {'SpaPhish': 632, 'SpearPhishMX': 1059, 'All': 1691},
 'phishing': {'SpaPhish': 711, 'SpearPhishMX': 1866, 'All': 2577},
 'All': {'SpaPhish': 1343, 'SpearPhishMX': 2925, 'All': 4268}}

In [6]:
# Encontrar las filas donde 'texto_modelo' o 'texto_defang' aún contienen IoCs vivos
filas_con_iocs = D[
D["texto_modelo"].map(ioc.iocs_vivos).map(bool) |
D["texto_defang"].map(ioc.iocs_vivos).map(bool)
].copy()

print(f"Número de filas con IoCs no defangueados: {len(filas_con_iocs)}")
if len(filas_con_iocs) > 0:
    # Mostrar la fuente, la clase y los IoCs detectados para diagnóstico
    filas_con_iocs["iocs_encontrados_modelo"] = filas_con_iocs["texto_modelo"].map(ioc.iocs_vivos)
    filas_con_iocs["iocs_encontrados_defang"] = filas_con_iocs["texto_defang"].map(ioc.iocs_vivos)
    display(filas_con_iocs[["fuente", "clase", "iocs_encontrados_modelo", "iocs_encontrados_defang", "texto"]].head(10))


Número de filas con IoCs no defangueados: 0


In [7]:
# Celda 6: RESUMEN
RESUMEN = {"entorno": ENV, "E1_esquema": {f: {k: v for k, v in e.items() if k != "mapeo"} | {"mapeo": e["mapeo"]} for f, e in E1.items()},
           "esperado_publicado": ESPERADO, "E2_idioma_pct": E2, "E3_duplicados": E3, "E4_urls_y_marcadores": E4, "E5_anios": E5,
           "exclusiones": bit, "final_grupo_x_clase": TABLA, "split_x_clase": SPLIT,
           "marcadores_por_fuente_pct": {g: {k: round(100 * float(sub.str.contains(v, regex=False).mean()), 1) for k, v in dp.MARCADORES.items()}
                                 for g, sub in D.groupby("grupo")["texto_modelo"]},
           "marcadores_crudos_residuales": int(D["texto_modelo"].str.contains(MARCADOR).sum())}
(BASE / "results/datos_es_v1.json").write_text(json.dumps(RESUMEN, indent=1, ensure_ascii=False, default=str))
print("===== RESUMEN (10) =====")
print(json.dumps(RESUMEN, ensure_ascii=False, default=str, separators=(",", ":")))

===== RESUMEN (10) =====
{"entorno":"colab","E1_esquema":{"SpaPhish":{"archivo":"Spaphish dataset - DiB.csv","filas":1395,"columnas":47,"etiquetas":{"1":731,"0":664},"mapeo":{"asunto":"subject","cuerpo":"body","etiqueta":"Label","fecha":"date","urls":"urls","clave":"hash"}},"SpearPhishMX":{"archivo":"SpearPhishMX_dataset_publico.csv","filas":3006,"columnas":17,"etiquetas":{"1":1891,"0":1115},"mapeo":{"asunto":"asunto","cuerpo":"cuerpo_sanitizado","etiqueta":"etiqueta","fecha":"fecha","urls":null,"clave":"hash"}}},"esperado_publicado":{"SpaPhish":{"filas":1395,"1":731,"0":664},"SpearPhishMX":{"filas":3006}},"E2_idioma_pct":{"SpaPhish":{"es":96.3,"en":2.2,"pt":0.6,"corto":0.2,"ca":0.0},"SpearPhishMX":{"es":97.4,"en":1.7,"pt":0.5,"corto":0.0,"ca":0.2}},"E3_duplicados":{"duplicados_intra":{"SpaPhish":0,"SpearPhishMX":2},"hashes_en_ambas_fuentes":1,"hashes_con_clases_en_conflicto":1},"E4_urls_y_marcadores":{"SpaPhish":{"pct_con_url":81.4,"pct_ioc_vivo_en_crudo":29.5,"marcadores_top":{"[NAME